# Sleep summary → Prism (worked example)

Turns the pipeline's **whole-recording** sleep export into GraphPad-Prism tables.
Companion to the by-hour example — same idea, but one number per animal per
recording instead of a time course.

**Input:** `sleep_whole_file.csv` from one or more `sleep_export/` folders. Each row
is one animal × sleep state, standardized over the whole recording.

**Two outputs:**
1. **Summary** (always) — one column per group, animals stacked down each column;
   a Prism *grouped column* table. One sheet per *state × metric*.
2. **Repeated measures** (when you list more than one timepoint) — rows = timepoint,
   columns = animals blocked by group; a Prism *repeated-measures* table.

Metrics map straight from the export: `Fraction` ← `frac_time`,
`Average_Duration` ← `mean_bout_sec`, `Bout_Frequency` ← `bouts_per_hour`.

In [1]:
import pandas as pd
from pathlib import Path
from openpyxl import Workbook

## 1. Point at your export(s)

`EXPORTS` maps a **timepoint label** to a `sleep_export/` folder. Use a single entry
for a one-off summary; add more entries (in the order you want them plotted) to also
get a repeated-measures workbook — the same animals recorded at each timepoint.

In [2]:
EXPORTS = {
    "Pre":     Path("/media/matt/Data_A/leah/Kyunghee - KCNT1/KCNT1_Pre/sleep_export"),
    "Post": Path("/media/matt/Data_A/leah/Kyunghee - KCNT1/KCNT1_Post/sleep_export"),
}

frames = []
for timepoint, export_dir in EXPORTS.items():
    w = pd.read_csv(export_dir / "sleep_whole_file.csv")
    w["timepoint"] = timepoint
    frames.append(w)
whole = pd.concat(frames, ignore_index=True)
whole.head()

,mouse_id,state,frac_time,n_bouts,mean_bout_sec,bouts_per_hour,n_hours,n_days,timepoint
0,KK0172,w,0.611574,829,127.478890,17.270833,48.0,2.0,Pre
1,KK0172,n,0.349907,830,72.848193,17.291667,48.0,2.0,Pre
2,KK0172,r,0.038519,133,50.045113,2.770833,48.0,2.0,Pre
3,KK0179,w,0.567662,830,118.183133,17.291667,48.0,2.0,Pre
4,KK0179,n,0.408403,830,85.026506,17.291667,48.0,2.0,Pre


Columns of interest: `mouse_id`, `state` (`w`/`n`/`r`), `frac_time`, `n_bouts`,
`mean_bout_sec`, `bouts_per_hour`, plus the `timepoint` we just added.

## 2. Assign each animal to a group

Same as the by-hour example: edit the dict for your project. Keys become the group
blocks in Prism; values are `mouse_id`s exactly as they appear in the export.

In [3]:
groups = {
    'A': ['KK0179', 'KK0186', 'KK0194', 'KK0202', 'KK0205', 'KK0207'],
    'B': ['KK0172', 'KK0184', 'KK0193', 'KK0198', 'KK0201', 'KK0206'],
}

animal_to_group = {a: g for g, animals in groups.items() for a in animals}
whole["group"] = whole["mouse_id"].map(animal_to_group)

unassigned = sorted(whole.loc[whole["group"].isna(), "mouse_id"].unique())
assert not unassigned, f"These animals are not in any group: {unassigned}"
print(f"{len(animal_to_group)} animals assigned across groups: {list(groups)}")

12 animals assigned across groups: ['A', 'B']


## 3. Config + helpers

`METRICS` maps each Prism sheet's metric name to a column in `sleep_whole_file.csv`;
`STATE_DISPLAY` maps the raw state code to its display name. The two writer functions
are the reusable piece — **copy this cell** into your own module to reuse them.

In [4]:
METRICS = {                       # sheet label -> column in sleep_whole_file.csv
    "Fraction": "frac_time",
    "Average_Duration": "mean_bout_sec",
    "Bout_Frequency": "bouts_per_hour",
}
STATE_DISPLAY = {"w": "Wake", "n": "NREM", "r": "REM"}


def write_summary_workbook(df, out_path, *, group_order, metrics=METRICS, states=STATE_DISPLAY):
    """Prism grouped-column table: one column per group, animal values stacked.

    If an animal has several rows (e.g. multiple timepoints) they are averaged, so
    this collapses to one summary value per animal."""
    wb = Workbook()
    wb.remove(wb.active)
    for label, col in metrics.items():
        for state_key, state_name in states.items():
            sub = df[df["state"] == state_key]
            group_vals = {
                g: sub.loc[sub["group"] == g].groupby("mouse_id")[col].mean().tolist()
                for g in group_order
            }
            ws = wb.create_sheet(f"{state_name}_{label}"[:31])
            for j, g in enumerate(group_order, start=1):
                ws.cell(row=1, column=j, value=g)
                for i, v in enumerate(group_vals[g], start=2):
                    ws.cell(row=i, column=j, value=None if pd.isna(v) else float(v))
    wb.save(out_path)
    return out_path


def write_repeated_measures_workbook(
    df, out_path, *, group_order, timepoint_order, metrics=METRICS, states=STATE_DISPLAY
):
    """Prism repeated-measures table: rows = timepoint, columns = animals blocked by
    group, with a merged group header row above the animal-id row."""
    wb = Workbook()
    wb.remove(wb.active)
    for label, col in metrics.items():
        for state_key, state_name in states.items():
            sub = df[df["state"] == state_key]
            group_animals = {
                g: sorted(sub.loc[sub["group"] == g, "mouse_id"].unique())
                for g in group_order
            }
            wide = sub.pivot_table(index="timepoint", columns="mouse_id", values=col)

            ws = wb.create_sheet(f"{state_name}_{label}"[:31])
            # header rows: merged group label (row 1) over its animal ids (row 2)
            c = 2
            for g in group_order:
                animals = group_animals[g]
                if not animals:
                    continue
                start = c
                for a in animals:
                    ws.cell(row=2, column=c, value=a)
                    c += 1
                ws.cell(row=1, column=start, value=g)
                if c - 1 > start:
                    ws.merge_cells(start_row=1, start_column=start, end_row=1, end_column=c - 1)
            # data rows: one per timepoint
            for r, tp in enumerate(timepoint_order, start=3):
                ws.cell(row=r, column=1, value=tp)
                c = 2
                for g in group_order:
                    for a in group_animals[g]:
                        v = wide.at[tp, a] if (tp in wide.index and a in wide.columns) else None
                        ws.cell(row=r, column=c, value=None if pd.isna(v) else float(v))
                        c += 1
            ws.freeze_panes = "B3"
    wb.save(out_path)
    return out_path

## 4. Write the summary workbook

In [5]:
OUT_DIR = list(EXPORTS.values())[0]  # write next to the first export
summary_path = write_summary_workbook(whole, OUT_DIR / "prism_sleep_summary.xlsx", group_order=list(groups))
print("Wrote", summary_path)

Wrote /media/matt/Data_A/leah/Kyunghee - KCNT1/KCNT1_Pre/sleep_export/prism_sleep_summary.xlsx


## 5. Write the repeated-measures workbook

Only meaningful with more than one timepoint in `EXPORTS`; otherwise this cell just
reports that it was skipped.

In [6]:
timepoint_order = list(EXPORTS)  # plotting order = the order you listed them

if len(timepoint_order) > 1:
    rm_path = write_repeated_measures_workbook(
        whole, OUT_DIR / "prism_sleep_repeated_measures.xlsx",
        group_order=list(groups), timepoint_order=timepoint_order,
    )
    print("Wrote", rm_path)
else:
    print("Only one timepoint in EXPORTS - skipping repeated-measures workbook. "
          "Add more entries to EXPORTS to enable it.")

Wrote /media/matt/Data_A/leah/Kyunghee - KCNT1/KCNT1_Pre/sleep_export/prism_sleep_repeated_measures.xlsx


Each sheet (`Wake_Fraction`, `NREM_Bout_Frequency`, …) drops straight into Prism:
the **summary** sheets are grouped column tables (paste each group column into a
Prism *Column* table), and the **repeated-measures** sheets are timepoint × animal
tables (paste into a *Grouped* / repeated-measures table).

To reuse this outside the notebook, copy the config + two writer functions from
step 3 into your own `.py` module.